In [1]:
import sys
sys.path.append('/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI')

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

from FaultGenerators.WeightFaultInjector import WeightFaultInjector
from BERCampaign import BERCampaign

In [2]:
class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool2d((1, 1))
        self.fc    = nn.Linear(16, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)

In [3]:
torch.manual_seed(42)
images = torch.randn(200, 3, 8, 8)
labels = torch.randint(0, 10, (200,))
dataset = TensorDataset(images, labels)
loader  = DataLoader(dataset, batch_size=50)

In [4]:
device   = torch.device('cpu')
network  = ToyNet().to(device)
injector = WeightFaultInjector(network)

campaign = BERCampaign(
    network=network,
    loader=loader,
    injector=injector,
    device=device,
    injection_levels=[1, 10, 50, 100],
    sampling_mode='constant',
    network_name='ToyNet',
    dataset_name='TOY',
    root='.',
    pilot_trials=10,
    max_trials=50,
    precision_e=0.01,
    confidence_t=2.576,
    seed=42,
)

In [5]:
print(f"layer_names   : {campaign.layer_names}")
print(f"layer_shapes  : {campaign.layer_shapes}")
print(f"layer_offsets : {campaign.layer_offsets}")
print(f"total_weights : {campaign.total_weights}")
print(f"total_bits    : {campaign.total_bits}")

layer_names   : ['conv1', 'conv2']
layer_shapes  : [(8, 3, 3, 3), (16, 8, 3, 3)]
layer_offsets : [  0 216]
total_weights : 1368
total_bits    : 43776


In [6]:
results = campaign.run()

Computing golden outputs...
Golden outputs saved to ./output/golden_output_ber/ToyNet_TOY_golden.pt
Golden accuracy: 0.0650



BER Campaign:  25%|██▌       | 1/4 [00:00<00:00,  5.93it/s]

[BERCampaign] level=1 | trials=10 | M=0.0000 | crit=0.0000 | std_M=0.0000 | half_width=0.0000


BER Campaign:  25%|██▌       | 1/4 [00:00<00:00,  5.93it/s]

[BERCampaign] level=10 | trials=10 | M=0.0000 | crit=0.1000 | std_M=0.0000 | half_width=0.0000


BER Campaign:  75%|███████▌  | 3/4 [00:00<00:00, 10.23it/s]

[BERCampaign] level=50 | trials=10 | M=0.0000 | crit=0.8000 | std_M=0.0000 | half_width=0.0000


BER Campaign: 100%|██████████| 4/4 [00:00<00:00,  8.02it/s]

[BERCampaign] level=100 | trials=14 | M=0.0000 | crit=1.0000 | std_M=0.0000 | half_width=0.0000


In [7]:
golden = list(results.values())[0]['golden_accuracy']
print(f'golden_accuracy = {golden:.4f}\n')

for level, r in results.items():
    print(f"injection_level = {level}")
    print(f"  mean masking ratio   = {r['mean_masking_ratio']:.4f}")
    print(f"  mean critical ratio  = {r['mean_critical_ratio']:.4f}")
    print(f"  mean faulty accuracy = {r['mean_faulty_accuracy']:.4f}")
    print(f"  sizing metric        = {r['sizing_metric']}")
    print(f"  n_trials             = {r['n_trials']}")
    print(f"  half_width           = {r['effective_half_width']:.4f}")
    print()

golden_accuracy = 0.0650

injection_level = 1
  mean masking ratio   = 0.0000
  mean critical ratio  = 0.0000
  mean faulty accuracy = 0.0650
  sizing metric        = accuracy
  n_trials             = 10
  half_width           = 0.0000

injection_level = 10
  mean masking ratio   = 0.0000
  mean critical ratio  = 0.1000
  mean faulty accuracy = 0.0670
  sizing metric        = accuracy
  n_trials             = 10
  half_width           = 0.0000

injection_level = 50
  mean masking ratio   = 0.0000
  mean critical ratio  = 0.8000
  mean faulty accuracy = 0.0825
  sizing metric        = accuracy
  n_trials             = 10
  half_width           = 0.0000

injection_level = 100
  mean masking ratio   = 0.0000
  mean critical ratio  = 1.0000
  mean faulty accuracy = 0.0907
  sizing metric        = accuracy
  n_trials             = 14
  half_width           = 0.0000

